# 18. 3D geometry and rendering — NeRF and 3D Gaussian Splatting

Only widths, ray/image counts, and sample counts are reduced. Training-time sampling behavior, SH degree, anisotropic covariance, compositing, and density-control branches are preserved instead of being renamed approximations.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(3)
device = torch.device("cpu")


## 1. NeRF field: 8 position layers, layer-4 skip, view-dependent color branch


In [ ]:
def positional_encoding(x, frequencies):
    encoded = [x]
    for frequency in frequencies:
        encoded.append(torch.sin(frequency * x))
        encoded.append(torch.cos(frequency * x))
    return torch.cat(encoded, -1)


class NeRFField(nn.Module):
    def __init__(self, hidden=32, position_frequencies=10, direction_frequencies=4):
        super().__init__()
        self.register_buffer("position_freq", 2.0 ** torch.arange(position_frequencies), persistent=False)
        self.register_buffer("direction_freq", 2.0 ** torch.arange(direction_frequencies), persistent=False)
        position_dim = 3 * (1 + 2 * position_frequencies)
        direction_dim = 3 * (1 + 2 * direction_frequencies)

        self.position_layers = nn.ModuleList()
        for index in range(8):
            if index == 0:
                input_dim = position_dim
            elif index == 4:
                input_dim = hidden + position_dim
            else:
                input_dim = hidden
            self.position_layers.append(nn.Linear(input_dim, hidden))

        self.density = nn.Linear(hidden, 1)
        self.feature = nn.Linear(hidden, hidden)
        self.view_hidden = nn.Linear(hidden + direction_dim, hidden // 2)
        self.color = nn.Linear(hidden // 2, 3)

    def forward(self, position, direction):
        encoded_position = positional_encoding(position, self.position_freq)
        h = encoded_position
        for index, layer in enumerate(self.position_layers):
            if index == 4:
                h = torch.cat([h, encoded_position], -1)
            h = F.relu(layer(h))
        density = F.relu(self.density(h)).squeeze(-1)
        feature = self.feature(h)
        encoded_direction = positional_encoding(direction, self.direction_freq)
        color_input = torch.cat([feature, encoded_direction], -1)
        color = torch.sigmoid(self.color(F.relu(self.view_hidden(color_input))))
        return color, density


coarse_field = NeRFField()
fine_field = NeRFField()
assert len(coarse_field.position_layers) == 8


## 2. Stratified coarse sampling + hierarchical PDF sampling


In [ ]:
def volume_render(color, density, depths):
    delta = depths[..., 1:] - depths[..., :-1]
    delta = torch.cat([delta, torch.full_like(depths[..., -1:], 1e10)], -1)
    alpha = 1 - torch.exp(-density * delta)
    transmittance = torch.cumprod(
        torch.cat([torch.ones_like(alpha[..., :1]), 1 - alpha + 1e-10], -1),
        -1,
    )[..., :-1]
    weights = transmittance * alpha
    rgb = (weights[..., None] * color).sum(-2)
    return rgb, weights


def stratified_depths(ray_count, samples, near, far, training, device):
    edges = torch.linspace(near, far, samples + 1, device=device)
    lower = edges[:-1]
    upper = edges[1:]
    if training:
        u = torch.rand(ray_count, samples, device=device)
    else:
        u = torch.full((ray_count, samples), 0.5, device=device)
    return lower[None] + (upper - lower)[None] * u


def sample_pdf(bins, weights, samples, training):
    weights = weights + 1e-5
    pdf = weights / weights.sum(-1, keepdim=True)
    cdf = torch.cumsum(pdf, -1)
    cdf = torch.cat([torch.zeros_like(cdf[..., :1]), cdf], -1)
    if training:
        u = torch.rand(*cdf.shape[:-1], samples, device=bins.device)
    else:
        u = torch.linspace(0, 1, samples, device=bins.device).expand(*cdf.shape[:-1], samples)
    indices = torch.searchsorted(cdf.contiguous(), u.contiguous(), right=True)
    below = (indices - 1).clamp_min(0)
    above = indices.clamp_max(cdf.size(-1) - 1)
    cdf_below = torch.gather(cdf, -1, below)
    cdf_above = torch.gather(cdf, -1, above)
    bins_below = torch.gather(bins, -1, below)
    bins_above = torch.gather(bins, -1, above)
    denom = (cdf_above - cdf_below).clamp_min(1e-5)
    t = (u - cdf_below) / denom
    return bins_below + t * (bins_above - bins_below)


def query(field, origins, directions, depths):
    points = origins[:, None] + directions[:, None] * depths[..., None]
    dirs = directions[:, None].expand_as(points)
    color, density = field(points.reshape(-1, 3), dirs.reshape(-1, 3))
    color = color.view(points.size(0), points.size(1), 3)
    density = density.view(points.size(0), points.size(1))
    return color, density


def render_nerf(origins, directions, coarse_samples=8, fine_samples=8, training=True):
    coarse_depths = stratified_depths(
        origins.size(0),
        coarse_samples,
        0.5,
        3.0,
        training,
        origins.device,
    )
    coarse_color, coarse_density = query(coarse_field, origins, directions, coarse_depths)
    coarse_rgb, coarse_weights = volume_render(coarse_color, coarse_density, coarse_depths)
    midpoints = 0.5 * (coarse_depths[..., 1:] + coarse_depths[..., :-1])
    fine_depths = sample_pdf(midpoints, coarse_weights[..., 1:-1].detach(), fine_samples, training)
    all_depths = torch.sort(torch.cat([coarse_depths, fine_depths], -1), -1).values
    fine_color, fine_density = query(fine_field, origins, directions, all_depths)
    fine_rgb, fine_weights = volume_render(fine_color, fine_density, all_depths)
    return coarse_rgb, fine_rgb, coarse_weights, fine_weights


origins = torch.zeros(4, 3)
directions = F.normalize(torch.randn(4, 3), -1)
coarse_rgb, fine_rgb, coarse_weights, fine_weights = render_nerf(origins, directions, training=True)
(coarse_rgb.square().mean() + fine_rgb.square().mean()).backward()
assert coarse_weights.size(-1) == 8
assert fine_weights.size(-1) == 16


## 3. 3DGS covariance and degree-3 spherical harmonics


In [ ]:
def quaternion_to_rotation(q):
    q = F.normalize(q, dim=-1)
    w, x, y, z = q.unbind(-1)
    rotation = torch.stack(
        [
            1 - 2 * (y.square() + z.square()),
            2 * (x * y - w * z),
            2 * (x * z + w * y),
            2 * (x * y + w * z),
            1 - 2 * (x.square() + z.square()),
            2 * (y * z - w * x),
            2 * (x * z - w * y),
            2 * (y * z + w * x),
            1 - 2 * (x.square() + y.square()),
        ],
        -1,
    )
    return rotation.view(-1, 3, 3)


def covariance_3d(log_scales, quaternion):
    scales = torch.exp(log_scales)
    rotation = quaternion_to_rotation(quaternion)
    linear = rotation @ torch.diag_embed(scales)
    return linear @ linear.transpose(-1, -2)


def sh_basis_degree3(direction):
    x, y, z = F.normalize(direction, dim=-1).unbind(-1)
    xx, yy, zz = x * x, y * y, z * z
    return torch.stack(
        [
            torch.full_like(x, 0.28209479177387814),
            -0.4886025119029199 * y,
            0.4886025119029199 * z,
            -0.4886025119029199 * x,
            1.0925484305920792 * x * y,
            -1.0925484305920792 * y * z,
            0.31539156525252005 * (3 * zz - 1),
            -1.0925484305920792 * x * z,
            0.5462742152960396 * (xx - yy),
            -0.5900435899266435 * y * (3 * xx - yy),
            2.890611442640554 * x * y * z,
            -0.4570457994644658 * y * (5 * zz - 1),
            0.3731763325901154 * z * (5 * zz - 3),
            -0.4570457994644658 * x * (5 * zz - 1),
            1.445305721320277 * z * (xx - yy),
            -0.5900435899266435 * x * (xx - 3 * yy),
        ],
        -1,
    )


def evaluate_sh_degree3(coefficients, direction):
    basis = sh_basis_degree3(direction)
    assert coefficients.size(1) == 16
    return torch.clamp(torch.einsum("nk,nkc->nc", basis, coefficients) + 0.5, min=0.0)


## 4. Differentiable projection and front-to-back alpha compositing


In [ ]:
def project_gaussians(means, covariance, height, width, fx=20.0, fy=20.0):
    depth = means[:, 2].clamp_min(0.2)
    u = fx * means[:, 0] / depth + width / 2
    v = fy * means[:, 1] / depth + height / 2
    means_2d = torch.stack([u, v], -1)
    means_2d.retain_grad()
    projected_covariance = []

    for index in range(means.size(0)):
        x, y, z = means[index]
        zero = torch.zeros((), device=means.device)
        jacobian = torch.stack(
            [
                torch.stack([fx / z, zero, -fx * x / z.square()]),
                torch.stack([zero, fy / z, -fy * y / z.square()]),
            ]
        )
        cov2 = jacobian @ covariance[index] @ jacobian.transpose(0, 1)
        cov2 = cov2 + 1e-4 * torch.eye(2, device=means.device)
        projected_covariance.append(cov2)

    return means_2d, torch.stack(projected_covariance), depth


def render_gaussians(means, log_scales, quaternion, opacity_logits, sh_coefficients, height=8, width=8):
    covariance = covariance_3d(log_scales, quaternion)
    means_2d, covariance_2d, depth = project_gaussians(means, covariance, height, width)
    yy, xx = torch.meshgrid(
        torch.arange(height, dtype=means.dtype),
        torch.arange(width, dtype=means.dtype),
        indexing="ij",
    )
    pixels = torch.stack([xx, yy], -1)
    colors = evaluate_sh_degree3(sh_coefficients, -means)
    opacity = torch.sigmoid(opacity_logits).squeeze(-1)
    image = torch.zeros(height, width, 3)
    transmittance = torch.ones(height, width)

    for gaussian_index in depth.argsort():
        offset = pixels - means_2d[gaussian_index]
        inverse_covariance = torch.linalg.inv(covariance_2d[gaussian_index])
        mahalanobis = torch.einsum("...i,ij,...j->...", offset, inverse_covariance, offset)
        gaussian = torch.exp(-0.5 * mahalanobis)
        alpha = (opacity[gaussian_index] * gaussian).clamp(0.0, 0.99)
        weight = transmittance * alpha
        image = image + weight[..., None] * colors[gaussian_index]
        transmittance = transmittance * (1 - alpha)

    return image, {"means_2d": means_2d, "depth": depth, "scales": torch.exp(log_scales)}


## 5. 3DGS adaptive density control — clone, stochastic split, prune, opacity reset


In [ ]:
@torch.no_grad()
def adaptive_density_control(
    means,
    log_scales,
    quaternion,
    opacity_logits,
    sh_coefficients,
    projected_gradient,
    gradient_threshold,
    percent_dense,
    scene_extent,
    max_screen_size=None,
    max_radii2d=None,
):
    scales = torch.exp(log_scales)
    max_scale = scales.max(-1).values
    high_gradient = projected_gradient >= gradient_threshold
    clone_mask = high_gradient & (max_scale <= percent_dense * scene_extent)
    split_mask = high_gradient & (max_scale > percent_dense * scene_extent)

    keep_mask = torch.sigmoid(opacity_logits).squeeze(-1) >= 0.005
    if max_screen_size is not None and max_radii2d is not None:
        keep_mask &= max_radii2d <= max_screen_size
        keep_mask &= max_scale <= 0.1 * scene_extent

    parent_keep = keep_mask & ~split_mask
    new_means = [means[parent_keep]]
    new_scales = [log_scales[parent_keep]]
    new_quaternion = [quaternion[parent_keep]]
    new_opacity = [opacity_logits[parent_keep]]
    new_sh = [sh_coefficients[parent_keep]]

    if clone_mask.any():
        new_means.append(means[clone_mask].clone())
        new_scales.append(log_scales[clone_mask].clone())
        new_quaternion.append(quaternion[clone_mask].clone())
        new_opacity.append(opacity_logits[clone_mask].clone())
        new_sh.append(sh_coefficients[clone_mask].clone())

    if split_mask.any():
        parent_means = means[split_mask]
        parent_scales = scales[split_mask]
        parent_rotation = quaternion_to_rotation(quaternion[split_mask])
        for _ in range(2):
            local_offset = torch.randn_like(parent_means) * parent_scales
            world_offset = torch.einsum("nij,nj->ni", parent_rotation, local_offset)
            new_means.append(parent_means + world_offset)
            new_scales.append(torch.log(parent_scales / 1.6))
            new_quaternion.append(quaternion[split_mask].clone())
            new_opacity.append(opacity_logits[split_mask].clone())
            new_sh.append(sh_coefficients[split_mask].clone())

    return {
        "means": torch.cat(new_means),
        "log_scales": torch.cat(new_scales),
        "quaternion": torch.cat(new_quaternion),
        "opacity_logits": torch.cat(new_opacity),
        "sh": torch.cat(new_sh),
        "cloned": int(clone_mask.sum()),
        "split": int(split_mask.sum()),
        "pruned": int((~keep_mask).sum()),
    }


@torch.no_grad()
def reset_opacity(opacity_logits, target=0.01):
    target_logit = math.log(target / (1 - target))
    opacity_logits.clamp_(max=target_logit)


count = 6
means = nn.Parameter(torch.randn(count, 3) * 0.2 + torch.tensor([0.0, 0.0, 2.0]))
log_scales = nn.Parameter(torch.log(torch.rand(count, 3) * 0.12 + 0.02))
quaternion = nn.Parameter(torch.randn(count, 4))
opacity_logits = nn.Parameter(torch.zeros(count, 1))
sh = nn.Parameter(torch.randn(count, 16, 3) * 0.05)
image, info = render_gaussians(means, log_scales, quaternion, opacity_logits, sh)
image.square().mean().backward()
projected_gradient = torch.linspace(0.0, 1.0, count)
refined = adaptive_density_control(
    means.detach(),
    log_scales.detach(),
    quaternion.detach(),
    opacity_logits.detach(),
    sh.detach(),
    projected_gradient,
    gradient_threshold=0.4,
    percent_dense=0.05,
    scene_extent=2.0,
    max_screen_size=100.0,
    max_radii2d=torch.zeros(count),
)
assert refined["sh"].shape[1] == 16
reset_copy = opacity_logits.detach().clone()
reset_opacity(reset_copy)


## Audit result

NeRF now includes training-time stratified/PDF randomness. 3DGS now uses degree-3 SH, stochastic anisotropic split offsets, scale/screen pruning conditions, clone/split/prune, and opacity reset without redefining these branches into deterministic stand-ins.
